In [43]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver

In [44]:
load_dotenv()

True

In [45]:
llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")

In [46]:
class jokeState(TypedDict):
    topic : str
    joke : str
    explanation : str

In [47]:
def joke_generator(state : jokeState):
    prompt = f"generate a funny joke on the given topic - {state['topic']}"
    response = llm.invoke(prompt).content
    return {"joke":response}

In [48]:
def joke_explanation(state : jokeState):
    prompt = f"explain the given joke - {state['joke']}"
    response = llm.invoke(prompt).content
    return {"explanation":response}

In [49]:
graph = StateGraph(jokeState)

graph.add_node("joke_generator",joke_generator)
graph.add_node("joke_explanation",joke_explanation)

graph.add_edge(START,"joke_generator")
graph.add_edge("joke_generator","joke_explanation")
graph.add_edge("joke_explanation",END)

In [50]:
checkpointer = InMemorySaver()

In [51]:
workflow = graph.compile(checkpointer=checkpointer)

In [52]:
config1 = {"configurable":{"thread_id":1}}

In [53]:
dog_initial_state = {"topic":"dog"}
dog_final_state = workflow.invoke(dog_initial_state,config=config1)

In [54]:
dog_final_state

{'topic': 'dog',
 'joke': 'Why did the dog eat all the money?\n\nBecause he heard his owner say, "I\'m going to *invest* in a new best friend!"',
 'explanation': 'This joke plays on a misunderstanding of the word "invest" and the literal-mindedness (and perhaps jealousy) of a dog.\n\nHere\'s the breakdown:\n\n1.  **What the owner meant:** When the owner says, "I\'m going to **invest** in a new best friend," they mean they\'re going to spend money (or time, effort) on getting a *new pet* (a puppy, another dog) or perhaps even on their *current dog* (e.g., better food, toys, training). "Invest" here is used in a broader sense of putting resources into something for a future benefit, often financial but sometimes emotional or practical.\n\n2.  **What the dog understood:** Dogs don\'t understand abstract financial concepts. The dog likely heard "invest," "money," and "new best friend." From a dog\'s very literal and self-preserving perspective:\n    *   If the owner is going to "invest" (p

In [55]:
config2 = {"configurable":{"thread_id":2}}

In [56]:
cat_initial_state = {"topic":"cat"}
cat_final_state = workflow.invoke(cat_initial_state,config=config2)

In [57]:
cat_final_state

{'topic': 'cat',
 'joke': 'Why do cats have nine lives?\n\nBecause it takes them at least eight just to decide if they want to go outside or stay in!',
 'explanation': 'This joke plays on two main ideas:\n\n1.  **The Myth of Nine Lives:** The popular saying that cats have nine lives is a common cultural belief, implying they are very resilient, agile, and can survive situations that would be fatal to other animals. The joke uses this as its setup.\n\n2.  **Cats\' Indecisiveness at Doors:** This is the core of the humor. Anyone who owns or has spent time with cats knows their peculiar habit of:\n    *   Demanding to go outside.\n    *   Getting to the door, smelling the air, and hesitating.\n    *   Stepping out, then immediately wanting back in.\n    *   Or staying in, then immediately wanting out again.\n    *   This back-and-forth, often taking a surprisingly long time, can be quite frustrating for their human companions.\n\n**How the Joke Works:**\n\nThe joke humorously suggests tha

In [58]:
workflow.get_state(config=config1)

StateSnapshot(values={'topic': 'dog', 'joke': 'Why did the dog eat all the money?\n\nBecause he heard his owner say, "I\'m going to *invest* in a new best friend!"', 'explanation': 'This joke plays on a misunderstanding of the word "invest" and the literal-mindedness (and perhaps jealousy) of a dog.\n\nHere\'s the breakdown:\n\n1.  **What the owner meant:** When the owner says, "I\'m going to **invest** in a new best friend," they mean they\'re going to spend money (or time, effort) on getting a *new pet* (a puppy, another dog) or perhaps even on their *current dog* (e.g., better food, toys, training). "Invest" here is used in a broader sense of putting resources into something for a future benefit, often financial but sometimes emotional or practical.\n\n2.  **What the dog understood:** Dogs don\'t understand abstract financial concepts. The dog likely heard "invest," "money," and "new best friend." From a dog\'s very literal and self-preserving perspective:\n    *   If the owner is g

In [59]:
workflow.get_state(config=config2)

StateSnapshot(values={'topic': 'cat', 'joke': 'Why do cats have nine lives?\n\nBecause it takes them at least eight just to decide if they want to go outside or stay in!', 'explanation': 'This joke plays on two main ideas:\n\n1.  **The Myth of Nine Lives:** The popular saying that cats have nine lives is a common cultural belief, implying they are very resilient, agile, and can survive situations that would be fatal to other animals. The joke uses this as its setup.\n\n2.  **Cats\' Indecisiveness at Doors:** This is the core of the humor. Anyone who owns or has spent time with cats knows their peculiar habit of:\n    *   Demanding to go outside.\n    *   Getting to the door, smelling the air, and hesitating.\n    *   Stepping out, then immediately wanting back in.\n    *   Or staying in, then immediately wanting out again.\n    *   This back-and-forth, often taking a surprisingly long time, can be quite frustrating for their human companions.\n\n**How the Joke Works:**\n\nThe joke humo

In [60]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'dog', 'joke': 'Why did the dog eat all the money?\n\nBecause he heard his owner say, "I\'m going to *invest* in a new best friend!"', 'explanation': 'This joke plays on a misunderstanding of the word "invest" and the literal-mindedness (and perhaps jealousy) of a dog.\n\nHere\'s the breakdown:\n\n1.  **What the owner meant:** When the owner says, "I\'m going to **invest** in a new best friend," they mean they\'re going to spend money (or time, effort) on getting a *new pet* (a puppy, another dog) or perhaps even on their *current dog* (e.g., better food, toys, training). "Invest" here is used in a broader sense of putting resources into something for a future benefit, often financial but sometimes emotional or practical.\n\n2.  **What the dog understood:** Dogs don\'t understand abstract financial concepts. The dog likely heard "invest," "money," and "new best friend." From a dog\'s very literal and self-preserving perspective:\n    *   If the owner is 

In [61]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'cat', 'joke': 'Why do cats have nine lives?\n\nBecause it takes them at least eight just to decide if they want to go outside or stay in!', 'explanation': 'This joke plays on two main ideas:\n\n1.  **The Myth of Nine Lives:** The popular saying that cats have nine lives is a common cultural belief, implying they are very resilient, agile, and can survive situations that would be fatal to other animals. The joke uses this as its setup.\n\n2.  **Cats\' Indecisiveness at Doors:** This is the core of the humor. Anyone who owns or has spent time with cats knows their peculiar habit of:\n    *   Demanding to go outside.\n    *   Getting to the door, smelling the air, and hesitating.\n    *   Stepping out, then immediately wanting back in.\n    *   Or staying in, then immediately wanting out again.\n    *   This back-and-forth, often taking a surprisingly long time, can be quite frustrating for their human companions.\n\n**How the Joke Works:**\n\nThe joke hum

<h5> Time Travel </h5>

In [65]:
workflow.get_state({"configurable":{"thread_id":2,"checkpoint_id":"1f1294d5-d10a-68b1-8000-808fb0270cc7"}})

StateSnapshot(values={'topic': 'cat'}, next=('joke_generator',), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f1294d5-d10a-68b1-8000-808fb0270cc7'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-03-26T19:52:47.578948+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1294d5-d107-6675-bfff-f992a5288ca8'}}, tasks=(PregelTask(id='6524cbc8-8187-8653-2a89-57f77885ceaa', name='joke_generator', path=('__pregel_pull', 'joke_generator'), error=None, interrupts=(), state=None, result={'joke': 'Why do cats have nine lives?\n\nBecause it takes them at least eight just to decide if they want to go outside or stay in!'}),), interrupts=())

In [66]:
workflow.invoke(None,{"configurable":{"thread_id":2,"checkpoint_id":"1f1294d5-d10a-68b1-8000-808fb0270cc7"}})

{'topic': 'cat',
 'joke': 'Why did the cat sit on the computer?\n\nTo keep an eye on the mouse!',
 'explanation': 'This is a classic pun! Here\'s why it\'s funny:\n\n1.  **Cats and Mice (Literal):** Cats are natural predators of mice (the small rodents). A cat\'s instinct is to "keep an eye on" a mouse to hunt it.\n\n2.  **Computer Mouse (Figurative/Pun):** A "mouse" is also the name of the pointing device you use with a computer.\n\nThe joke plays on the double meaning of the word "mouse." You expect the answer to relate to an actual animal mouse, but the punchline cleverly refers to the computer accessory, creating a silly image of a cat "guarding" a piece of hardware.\n\n**In short:** The cat is metaphorically "keeping an eye" on the computer mouse, just as it would an actual mouse.'}

In [67]:
list(workflow.get_state_history(config=config2))

[StateSnapshot(values={'topic': 'cat', 'joke': 'Why did the cat sit on the computer?\n\nTo keep an eye on the mouse!', 'explanation': 'This is a classic pun! Here\'s why it\'s funny:\n\n1.  **Cats and Mice (Literal):** Cats are natural predators of mice (the small rodents). A cat\'s instinct is to "keep an eye on" a mouse to hunt it.\n\n2.  **Computer Mouse (Figurative/Pun):** A "mouse" is also the name of the pointing device you use with a computer.\n\nThe joke plays on the double meaning of the word "mouse." You expect the answer to relate to an actual animal mouse, but the punchline cleverly refers to the computer accessory, creating a silly image of a cat "guarding" a piece of hardware.\n\n**In short:** The cat is metaphorically "keeping an eye" on the computer mouse, just as it would an actual mouse.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1294f2-e38b-6d1b-8002-e6c441ef87fd'}}, metadata={'source': 'loop', 'step': 2, 'parents'

<h5>update state </h5>

In [68]:
workflow.get_state({"configurable":{"thread_id":2,"checkpoint_id":"1f1294d5-d107-6675-bfff-f992a5288ca8"}})

StateSnapshot(values={}, next=('__start__',), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f1294d5-d107-6675-bfff-f992a5288ca8'}}, metadata={'source': 'input', 'step': -1, 'parents': {}}, created_at='2026-03-26T19:52:47.577662+00:00', parent_config=None, tasks=(PregelTask(id='31ff4e71-53b4-94fd-4be0-c8847bd52e4e', name='__start__', path=('__pregel_pull', '__start__'), error=None, interrupts=(), state=None, result={'topic': 'cat'}),), interrupts=())

In [81]:
workflow.update_state({"configurable":{"thread_id":2,"checkpoint_id":"1f1294d5-d107-6675-bfff-f992a5288ca8",'checkpoint_ns': ''}},{'topic':'elephant'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f129515-188b-6895-8000-84340172be24'}}

In [83]:
list(workflow.get_state_history(config=config2))

[StateSnapshot(values={'topic': 'elephant'}, next=('joke_generator',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f129515-188b-6895-8000-84340172be24'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-03-26T20:21:06.220046+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1294d5-d107-6675-bfff-f992a5288ca8'}}, tasks=(PregelTask(id='c6673e06-65f3-349b-58b0-7855d024acc9', name='joke_generator', path=('__pregel_pull', 'joke_generator'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'cat', 'joke': 'Why did the cat sit on the computer?\n\nTo keep an eye on the mouse!', 'explanation': 'This is a classic pun! Here\'s why it\'s funny:\n\n1.  **Cats and Mice (Literal):** Cats are natural predators of mice (the small rodents). A cat\'s instinct is to "keep an eye on" a mouse to hunt it.\n\n2.  **Computer Mouse (Figurative/Pun

In [87]:
workflow.invoke(None,{"configurable":{"thread_id":"2","checkpoint_id":"1f129515-188b-6895-8000-84340172be24"}})

{'topic': 'elephant',
 'joke': "Why don't elephants play hide-and-seek?\n\nBecause no matter where they hide, their trunk always gives them away!",
 'explanation': 'This joke plays on the obvious physical characteristics of an elephant and the rules of the game hide-and-seek.\n\nHere\'s the breakdown:\n\n1.  **Hide-and-Seek Rules:** The goal of hide-and-seek is to be completely concealed so that the "seeker" cannot find you. If any part of you is visible, you\'re "given away" and found.\n2.  **Elephant Characteristics:** Elephants are enormous animals. More specifically, they have a very long, prominent, and distinctive **trunk**.\n3.  **The Humor:** The joke creates a funny image of an elephant trying to hide its massive body, perhaps behind a tree or a bush. Even if it managed to hide its main body, its long trunk would almost inevitably stick out, giving away its location.\n\nThe humor comes from the silly image and the inescapable problem that an elephant\'s most defining feature (

In [89]:
list(workflow.get_state_history(config=config2))

[StateSnapshot(values={'topic': 'elephant', 'joke': "Why don't elephants play hide-and-seek?\n\nBecause no matter where they hide, their trunk always gives them away!", 'explanation': 'This joke plays on the obvious physical characteristics of an elephant and the rules of the game hide-and-seek.\n\nHere\'s the breakdown:\n\n1.  **Hide-and-Seek Rules:** The goal of hide-and-seek is to be completely concealed so that the "seeker" cannot find you. If any part of you is visible, you\'re "given away" and found.\n2.  **Elephant Characteristics:** Elephants are enormous animals. More specifically, they have a very long, prominent, and distinctive **trunk**.\n3.  **The Humor:** The joke creates a funny image of an elephant trying to hide its massive body, perhaps behind a tree or a bush. Even if it managed to hide its main body, its long trunk would almost inevitably stick out, giving away its location.\n\nThe humor comes from the silly image and the inescapable problem that an elephant\'s mos